# iSE Challenge 2026 - Colab Indexing

Run these cells on a Colab GPU runtime. For H100/A100, the notebook uses local Whisper `large-v3` for audio transcription and keeps VLM image parsing off during indexing to avoid API/quota burn.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%%bash
set -e
cd /content
rm -rf ise-challenge-2026
git clone "https://github.com/ManhTanTran/ise-challenge-2026.git" ise-challenge-2026
cd ise-challenge-2026
git status --short --branch
ls -la


In [ ]:
%%bash
set -e
cd /content/ise-challenge-2026
apt-get update
apt-get install -y ffmpeg tesseract-ocr tesseract-ocr-vie libreoffice
pip install -r requirements.txt
pip install -U openai-whisper sentence-transformers faiss-cpu
python - <<'PY'
import importlib.util as u
for name in ['whisper', 'sentence_transformers', 'faiss']:
    print(name, bool(u.find_spec(name)))
PY


In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
print(result.stdout or result.stderr)


In [ ]:
from pathlib import Path

# If auto-detection picks the wrong folder, edit DATA_LAKE_PATH manually below.
candidates = sorted(Path('/content/drive/MyDrive').rglob('[iSE Summer Challenge 2026] Data Lake'))
print('Candidates:')
for i, path in enumerate(candidates):
    file_count = sum(1 for child in path.rglob('*') if child.is_file())
    print(i, file_count, path)

DATA_LAKE_PATH = None
for path in candidates:
    nested = path / '[iSE Summer Challenge 2026] Data Lake'
    if nested.exists():
        path = nested
    file_count = sum(1 for child in path.rglob('*') if child.is_file())
    if file_count > 100:
        DATA_LAKE_PATH = path
        break

if DATA_LAKE_PATH is None:
    raise RuntimeError('Could not auto-detect the data lake. Add the Drive shortcut to My Drive, then rerun this cell.')

print('Using DATA_LAKE_PATH =', DATA_LAKE_PATH)
print('Files =', sum(1 for child in DATA_LAKE_PATH.rglob('*') if child.is_file()))


In [ ]:
import os
from pathlib import Path

os.chdir('/content/ise-challenge-2026')
os.environ['PYTHONUTF8'] = '1'
os.environ['ISE_IMAGE_PARSE_VLM'] = 'off'
os.environ['ISE_TRANSCRIPTION_PROVIDER'] = 'local'
os.environ['ISE_LOCAL_WHISPER_MODEL'] = 'large-v3'  # H100/A100: large-v3; L4/T4: consider turbo
os.environ['WHISPER_CACHE_DIR'] = '/content/drive/MyDrive/ise-challenge-index/whisper_model_cache'
os.environ['ISE_TRANSCRIPT_CACHE_DIR'] = '/content/drive/MyDrive/ise-challenge-index/transcript_cache'

from approaches.approach_3_agentic_rag.config import get_config
from approaches.approach_3_agentic_rag.indexing.build import build_indexes

work = Path('/content/drive/MyDrive/ise-challenge-index/full_data_index_colab_large_v3')
work.mkdir(parents=True, exist_ok=True)

cfg = get_config()
manifest, chunks, vector, bm25 = build_indexes(DATA_LAKE_PATH, work, config=cfg, rebuild=True)

print('INDEX_DONE')
print('manifest', len(manifest))
print('chunks', len(chunks))
print('vector_kind', vector.meta.get('kind'))
print('vector_model', vector.meta.get('model_name'))
print('work_dir', work.resolve())


## Fast fallback: skip audio transcription

Use this only if full Whisper transcription is still too slow. It indexes documents/tables/images first and marks audio for later transcription.


In [ ]:
import os
from pathlib import Path

os.chdir('/content/ise-challenge-2026')
os.environ['PYTHONUTF8'] = '1'
os.environ['ISE_IMAGE_PARSE_VLM'] = 'off'

from approaches.approach_3_agentic_rag.config import get_config
from approaches.approach_3_agentic_rag.indexing.build import build_indexes
from approaches.approach_3_agentic_rag.shared_src.file_readers import ReadResult
import approaches.approach_3_agentic_rag.shared_src.file_readers as fr

fr.read_audio = lambda p: ReadResult(str(p), 'audio', content='', metadata={'needs_audio_transcription': True}, error='Skipped during fast indexing')

work = Path('/content/drive/MyDrive/ise-challenge-index/full_data_index_fast_colab')
work.mkdir(parents=True, exist_ok=True)

cfg = get_config()
manifest, chunks, vector, bm25 = build_indexes(DATA_LAKE_PATH, work, config=cfg, rebuild=True)

print('INDEX_DONE_FAST')
print('manifest', len(manifest))
print('chunks', len(chunks))
print('vector_kind', vector.meta.get('kind'))
print('vector_model', vector.meta.get('model_name'))
print('work_dir', work.resolve())
